# Estatísticas Landsat e WorldCover dos setores urbanos

Este notebook seleciona setores urbanos, com renda válida e não pequenos. WorldCover limita apenas a composição do setor; ele não mascara o NDVI para pixels construídos.

Este notebook calcula estatísticas Landsat e WorldCover para a base urbana escolhida. **Entradas:** malha/renda preparada e coleções Earth Engine. **Saídas:** `data/processed/sector_stats_urban.csv`. **Posição:** prepara as variáveis de cobertura e NDVI usadas no Passo 2 do notebook 06.


In [ ]:
# A máscara restringe a tabela a setores com renda válida, urbanos e não pequenos para esta saída urbana.

import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"
ee.Initialize(project="fourth-carport-512220-f2")
sectors = gpd.read_file(PROCESSED / "sectors_income.gpkg")
urban = sectors.loc[sectors["has_income"] & ~sectors["is_rural"] & ~sectors["small_sector"]].copy()
urban["sector_id"] = urban["sector_id"].astype("string").str.zfill(15)
urban_wgs84 = urban.to_crs("EPSG:4326")
sector_fc = geemap.gdf_to_ee(urban_wgs84[["sector_id", "geometry"]])
roi = ee.Geometry(urban_wgs84.geometry.union_all().__geo_interface__)
print("Urban sectors:", len(urban))

## Landsat Collection 2 Level 2

A máscara QA_PIXEL exclui fill, nuvem dilatada, cirrus, nuvem, sombra de nuvem e água. O NDVI usa todos os pixels válidos restantes dentro de cada setor selecionado.

Selecionamos Landsat 8/9 Level 2 e aplicamos a máscara de qualidade antes de reduzir as bandas por setor. A escala de 30 m é a escala usada nas estatísticas.


In [ ]:
# O filtro temporal seleciona agosto e setembro de 2026, mantendo o período usado no projeto.

landsat = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
    .filterBounds(roi).filterDate("2026-08-01", "2026-10-01"))

def prepare(image):
    qa = image.select("QA_PIXEL")
    excluded = (1 << 0) | (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4) | (1 << 7)
    clear_land = qa.bitwiseAnd(excluded).eq(0)
    saturated = image.select("QA_RADSAT").bitwiseAnd((1 << 3) | (1 << 4)).eq(0)
    red = image.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = image.select("SR_B5").multiply(0.0000275).add(-0.2)
    valid = clear_land.And(saturated).And(red.gte(0)).And(nir.gte(0))
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi").updateMask(valid).updateMask(nir.add(red).neq(0))
    lst = image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("lst_c").updateMask(clear_land)
    return lst.addBands(ndvi).copyProperties(image, ["system:time_start"])

prepared = landsat.map(prepare)
dates = ee.List(prepared.aggregate_array("system:time_start")).map(lambda t: ee.Date(t).format("YYYY-MM-dd")).distinct()
def daily_image(date_string):
    date = ee.Date(date_string)
    return prepared.filterDate(date, date.advance(1, "day")).mosaic().set("system:time_start", date.millis())
daily = ee.ImageCollection.fromImages(dates.map(daily_image))
composite = daily.median().clip(roi)
composite = composite.addBands(daily.select("lst_c").count().rename("lst_valid_dates"))
composite = composite.addBands(daily.select("ndvi").count().rename("ndvi_valid_dates"))
print("Landsat scenes:", landsat.size().getInfo(), "acquisition dates:", daily.size().getInfo())

## ESA WorldCover 2021

A coleção é `ESA/WorldCover/v200`. As frações das classes usam a banda `Map` a 10 m e são exportadas como colunas separadas de porcentagem.

Usamos a banda `Map` em 10 m para calcular frações de classes. WorldCover descreve a composição do setor; não é aplicado como máscara de pixels de NDVI.


In [ ]:
# A banda Map de 10 m fornece a contagem de classes de cobertura do solo por setor.

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").clip(roi)
classes = {10: "tree", 20: "shrub", 30: "grass", 40: "cropland", 50: "built", 60: "bare", 70: "snow_ice", 80: "water", 90: "wetland", 95: "mangrove", 100: "moss_lichen"}
print("Collection:", "ESA/WorldCover/v200")
print("Class columns:", ["pct_" + name for name in classes.values()])

## Estatísticas por setor e exportação CSV

As estatísticas Landsat usam 30 m. `n_valid_pixels` conta pixels NDVI válidos no composto; os pixels térmicos válidos são informados separadamente. A fração por limiar de NDVI usa a mesma máscara válida de NDVI.

Agregamos NDVI, temperatura e cobertura do solo por setor e exportamos colunas nomeadas em inglês para o próximo notebook.


In [ ]:
# A escala de 30 m mantém a mesma escala das estatísticas Landsat por setor.

def summarize(feature):
    geometry = feature.geometry()
    ndvi = composite.select("ndvi")
    lst = composite.select("lst_c")
    ndvi_stats = ndvi.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.median(), sharedInputs=True).combine(ee.Reducer.percentile([90]), sharedInputs=True).combine(ee.Reducer.count(), sharedInputs=True), geometry=geometry, scale=30, maxPixels=1e8, tileScale=4)
    lst_stats = lst.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.median(), sharedInputs=True).combine(ee.Reducer.count(), sharedInputs=True), geometry=geometry, scale=30, maxPixels=1e8, tileScale=4)
    green = ndvi.gt(0.3).reduceRegion(ee.Reducer.mean(), geometry=geometry, scale=30, maxPixels=1e8, tileScale=4)
    wc_hist = ee.Dictionary(worldcover.reduceRegion(ee.Reducer.frequencyHistogram(), geometry=geometry, scale=10, maxPixels=1e8, tileScale=4).get("Map"))
    wc_total = ee.Number(wc_hist.values().reduce(ee.Reducer.sum()))
    props = {"ndvi_mean": ndvi_stats.get("ndvi_mean"), "ndvi_median": ndvi_stats.get("ndvi_median"), "ndvi_p90": ndvi_stats.get("ndvi_p90"), "fraction_ndvi_gt_0_3": green.get("ndvi"), "lst_mean_c": lst_stats.get("lst_c_mean"), "lst_median_c": lst_stats.get("lst_c_median"), "n_valid_pixels": ndvi_stats.get("ndvi_count"), "n_valid_lst_pixels": lst_stats.get("lst_c_count"), "lst_valid_dates": composite.select("lst_valid_dates").reduceRegion(ee.Reducer.mean(), geometry=geometry, scale=30, maxPixels=1e8, tileScale=4).get("lst_valid_dates"), "ndvi_valid_dates": composite.select("ndvi_valid_dates").reduceRegion(ee.Reducer.mean(), geometry=geometry, scale=30, maxPixels=1e8, tileScale=4).get("ndvi_valid_dates")}
    for code, name in classes.items():
        props["pct_" + name] = ee.Number(wc_hist.get(str(code), 0)).divide(wc_total).multiply(100)
    return feature.set(props)

stats_gdf = geemap.ee_to_gdf(sector_fc.map(summarize))
stats = stats_gdf.drop(columns="geometry")
stats.to_csv(PROCESSED / "sector_stats_urban.csv", index=False)
print("Saved sectors:", len(stats))

## Verificação da exportação

Conferimos a tabela exportada e suas colunas para evitar que o Passo 2 leia uma saída incompleta.


In [ ]:
exported = pd.read_csv(PROCESSED / "sector_stats_urban.csv")
print("Number of exported sectors:", len(exported))
print("NaN count by column:")
display(exported.isna().sum().to_frame("n_nan"))
display(exported.head())

## Conclusão da etapa

A etapa gera as frações WorldCover e as métricas Landsat para os setores urbanos selecionados. Esses campos permitem comparar cobertura do solo e NDVI entre os quintis no notebook 06.
